# Monthly Retail Revenue Forecasting

## Project Overview

This project forecasts **monthly retail revenue** over a 3-month horizon using statistical,
ML, and AutoML methods. We generate a synthetic dataset spanning 8 years of monthly revenue
with trend, yearly seasonality, and promotional spikes.

**Models**: Naive baselines, LazyPredict (tabular), FLAML AutoML, StatsForecast (AutoARIMA, AutoETS).

**Forecast horizon**: 3 months ahead.

## Learning Objectives

1. Understand monthly time-series patterns (trend, 12-month seasonality)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features at monthly granularity
4. Use LazyPredict for quick tabular regression benchmarking
5. Apply FLAML AutoML with restricted estimator list
6. Use StatsForecast for classical forecasting
7. Compare approaches using MAE / RMSE / MAPE

## Problem Statement

Given historical monthly revenue data, predict revenue for the next **3 months**.
Monthly forecasts drive quarterly budget planning, cash flow management, and strategic decisions.

## Why This Project Matters

Monthly revenue forecasting is fundamental for financial planning, investor reporting,
and resource allocation. Accurate forecasts reduce budget surprises and improve decision-making.

## Dataset Overview

Synthetic dataset:
- ~96 months (8 years) of monthly revenue
- Upward trend
- 12-month seasonal cycle (holiday peaks in Nov/Dec)
- Random noise

| Column | Description |
|--------|-------------|
| `date` | First day of month |
| `revenue` | Monthly total revenue ($) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.

## Configuration / Constants

In [3]:
SEED = 42
N_MONTHS = 96
FORECAST_HORIZON = 3
VAL_MONTHS = 3
TEST_MONTHS = 3
SEASON_LENGTH = 12
np.random.seed(SEED)
print(f"Config: {N_MONTHS} months, horizon={FORECAST_HORIZON}, season={SEASON_LENGTH}")

Config: 96 months, horizon=3, season=12


## Dataset Generation

In [4]:
dates = pd.date_range(start='2016-01-01', periods=N_MONTHS, freq='MS')
t = np.arange(N_MONTHS)

trend = 500000 + 5000 * t
seasonal = 80000 * np.sin(2 * np.pi * (t - 3) / 12)  # peak around month 9-10

# Holiday boost Nov/Dec
holiday = np.zeros(N_MONTHS)
for i in range(N_MONTHS):
    m = dates[i].month
    if m == 11:
        holiday[i] = 100000
    elif m == 12:
        holiday[i] = 150000

noise = np.random.normal(0, 30000, N_MONTHS)
revenue = trend + seasonal + holiday + noise
revenue = np.maximum(revenue, 100000)

df = pd.DataFrame({'date': dates, 'revenue': revenue})
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

Dataset shape: (96, 2)
Date range: 2016-01-01 00:00:00 to 2023-12-01 00:00:00


,date,revenue
0,2016-01-01,434901.424590
1,2016-02-01,431570.038662
2,2016-03-01,489430.656143
3,2016-04-01,560690.895692
4,2016-05-01,552975.398758


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert df['revenue'].min() > 0, "Negative revenue"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_MONTHS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(df['date'], df['revenue'], 'o-', markersize=3)
axes[0, 0].set_title('Monthly Revenue Over Time')

axes[0, 1].hist(df['revenue'], bins=25, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Revenue Distribution')

# Yearly aggregation
df_yearly = df.set_index('date').resample('YE').sum()
axes[1, 0].bar(df_yearly.index.year, df_yearly['revenue'], alpha=0.7)
axes[1, 0].set_title('Yearly Total Revenue')

# Month-of-year boxplot
df['month'] = df['date'].dt.month
df.boxplot(column='revenue', by='month', ax=axes[1, 1])
axes[1, 1].set_title('Revenue by Month')
plt.suptitle('')

plt.tight_layout()
plt.savefig('eda_monthly_revenue.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA plots saved.")

EDA plots saved.


## Target Analysis

In [7]:
print("Revenue Statistics:")
print(df['revenue'].describe())
print(f"\nCV: {df['revenue'].std() / df['revenue'].mean():.3f}")
print(f"Skewness: {df['revenue'].skew():.3f}")

Revenue Statistics:
count    9.600000e+01
mean     7.549857e+05
std      1.602331e+05
min      4.315700e+05
25%      6.261855e+05
50%      7.530420e+05
75%      9.009065e+05
max      1.053341e+06
Name: revenue, dtype: float64

CV: 0.212
Skewness: -0.039


## Train / Validation / Test Split Strategy

Time-aware split:
- **Train**: first 90 months
- **Validation**: 3 months
- **Test**: last 3 months

In [8]:
train = df.iloc[:-(VAL_MONTHS + TEST_MONTHS)].copy()
val = df.iloc[-(VAL_MONTHS + TEST_MONTHS):-(TEST_MONTHS)].copy()
test = df.iloc[-(TEST_MONTHS):].copy()

print(f"Train: {len(train)} months")
print(f"Val:   {len(val)} months")
print(f"Test:  {len(test)} months")

Train: 90 months
Val:   3 months
Test:  3 months


## Preprocessing Strategy

Monthly data needs minimal preprocessing. Tree-based models handle raw features well.
We engineer temporal features in the next section.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print("Data ready.")

Data ready.


## Feature Engineering

- **Lags**: 1, 3, 6, 12 months
- **Rolling**: mean/std over 3, 6, 12 month windows
- **Calendar**: month, quarter, year

In [10]:
def create_features(data):
    d = data.copy()
    d['month'] = d['date'].dt.month
    d['quarter'] = d['date'].dt.quarter
    d['year'] = d['date'].dt.year

    for lag in [1, 3, 6, 12]:
        d[f'lag_{lag}'] = d['revenue'].shift(lag)
    for w in [3, 6, 12]:
        d[f'rolling_mean_{w}'] = d['revenue'].shift(1).rolling(w).mean()
        d[f'rolling_std_{w}'] = d['revenue'].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full)
df_feat = df_feat.dropna().reset_index(drop=True)
print(f"Feature matrix: {df_feat.shape}")

Feature matrix: (84, 15)


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{name:30s} | MAE: {mae:12.1f} | RMSE: {rmse:12.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []

naive_pred = np.full(TEST_MONTHS, train['revenue'].iloc[-1])
results.append(calc_metrics(test['revenue'].values, naive_pred, "Naive (Last Value)"))

# Seasonal naive: same month last year
test_start = len(df_full) - TEST_MONTHS
seasonal_pred = df_full['revenue'].values[test_start - SEASON_LENGTH:test_start - SEASON_LENGTH + TEST_MONTHS]
results.append(calc_metrics(test['revenue'].values, seasonal_pred, "Seasonal Naive (12m)"))

Naive (Last Value)             | MAE:      34606.9 | RMSE:      44728.5 | MAPE:  3.56%
Seasonal Naive (12m)           | MAE:      24999.8 | RMSE:      29287.7 | MAPE:  2.55%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

feature_cols = [c for c in df_feat.columns if c not in ['date', 'revenue']]
cutoff_test = len(df_feat) - TEST_MONTHS
cutoff_val = cutoff_test - VAL_MONTHS

X_tr = df_feat.iloc[:cutoff_val][feature_cols]
y_tr = df_feat.iloc[:cutoff_val]['revenue']
X_va = df_feat.iloc[cutoff_val:cutoff_test][feature_cols]
y_va = df_feat.iloc[cutoff_val:cutoff_test]['revenue']

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                            Adjusted R-Squared    R-Squared          RMSE  Time Taken
Model                                                                                
MLPRegressor                        193.826060 -1059.543330  1.022151e+06    0.071289
LinearSVR                           193.819997 -1059.509982  1.022135e+06    0.009463
PassiveAggressiveRegressor          165.555376  -904.054568  9.442525e+05    0.016484
KernelRidge                         113.556587  -618.061226  7.809398e+05    0.010355
GaussianProcessRegressor            105.765662  -575.211141  7.534277e+05    0.012395
DummyRegressor                       13.259316   -66.426237  2.577301e+05    0.006069
SVR                                  12.372214   -61.547175  2.482302e+05    0.008815
QuantileRegressor                    12.351248   -61.431861  2.480013e+05    0.032790
ElasticNetCV                         12.215589   -60.685740  2.465149e+05    0.042130
NuSVR                            

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)

flaml_val_pred = automl.predict(X_va)
print(f"FLAML best: {automl.best_estimator}")
results.append(calc_metrics(y_va.values, flaml_val_pred, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[cutoff_test:][feature_cols]
y_te = df_feat.iloc[cutoff_test:]['revenue']
flaml_test_pred = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test_pred, f"FLAML Test ({automl.best_estimator})"))

FLAML best: extra_tree
FLAML (extra_tree)             | MAE:      72889.6 | RMSE:      76899.3 | MAPE:  7.07%
FLAML Test (extra_tree)        | MAE:      56434.8 | RMSE:      63149.7 | MAPE:  5.59%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', 'revenue']].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'store1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-(TEST_MONTHS)]

sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH), SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq='MS', n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_MONTHS).reset_index()

y_test_sf = df_full['revenue'].values[-TEST_MONTHS:]
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))

print("StatsForecast complete.")

SF AutoARIMA                   | MAE:      24101.0 | RMSE:      30810.9 | MAPE:  2.38%
SF AutoETS                     | MAE:      24459.8 | RMSE:      30745.7 | MAPE:  2.41%
SF SeasonalNaive               | MAE:      24999.8 | RMSE:      29287.7 | MAPE:  2.55%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                  model          MAE         RMSE     MAPE
           SF AutoARIMA 24100.999436 30810.935221 2.376891
             SF AutoETS 24459.828034 30745.658420 2.414418
   Seasonal Naive (12m) 24999.783542 29287.671385 2.547065
       SF SeasonalNaive 24999.783542 29287.671385 2.547065
     Naive (Last Value) 34606.918542 44728.537086 3.563475
FLAML Test (extra_tree) 56434.780055 63149.723821 5.593345
     FLAML (extra_tree) 72889.639611 76899.271392 7.066204

Top 3:
               model          MAE         RMSE     MAPE
        SF AutoARIMA 24100.999436 30810.935221 2.376891
          SF AutoETS 24459.828034 30745.658420 2.414418
Seasonal Naive (12m) 24999.783542 29287.671385 2.547065


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test['revenue'].values, 'ko-', label='Actual', markersize=6)
ax.plot(test['date'].values, flaml_test_pred, 's--', label=f'FLAML ({automl.best_estimator})', markersize=6)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', markersize=6)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test['revenue'].values - flaml_test_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=8, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.1f}, Std: {np.std(errors):.1f}")

Mean residual: 56434.8, Std: 28337.3


## Interpretation and Business Insight

- Revenue shows strong upward trend and clear Nov/Dec holiday peaks
- Seasonal naive is competitive due to consistent yearly patterns
- FLAML captures lag relationships for short-term forecasts
- 3-month horizon aligns with quarterly business planning cycles

## Limitations

1. Synthetic data — real revenue has more complex drivers
2. Only 3-month horizon — longer forecasts need different approaches
3. No external regressors (marketing spend, economic indicators)
4. Single revenue stream — real businesses have multiple product lines

## How to Improve This Project

1. Use real retail revenue datasets
2. Add marketing/promotion and macroeconomic features
3. Try hierarchical forecasting across product categories
4. Add probabilistic forecasts with prediction intervals
5. Use Chronos-Bolt/TimesFM foundation models

## Production Considerations

- Monthly retraining cadence
- Monitor MAPE drift over time
- Integrate with financial planning systems
- Version models with MLflow

## Common Mistakes

1. Using future months' data in lag features
2. Not accounting for holiday effects in monthly data
3. Over-fitting on small monthly datasets (few data points)
4. Ignoring the trend when using seasonal decomposition

## Mini Challenge / Exercises

1. Extend to 6-month forecast horizon — how does accuracy degrade?
2. Add a marketing spend feature and measure impact
3. Try Prophet and compare with StatsForecast results
4. Build a simple exponential smoothing model from scratch

## Final Summary / Key Takeaways

1. Monthly revenue forecasting requires accounting for both trend and seasonality
2. With only ~96 data points, simple models often outperform complex ones
3. Seasonal naive is a surprisingly strong baseline for monthly data
4. Combining statistical (AutoARIMA) and ML (FLAML) approaches gives robust forecasts
5. Short horizons (3 months) are more reliable than long-term forecasts

In [18]:
import json
metrics = {
    'project': 'Monthly Retail Revenue Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Monthly Retail Revenue Forecasting — Complete ===")

Metrics saved.

=== Monthly Retail Revenue Forecasting — Complete ===
